# Replicon/ServiceNow Timesheet Reconciliation

## Setup

In [ ]:
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import reconciliation as rec

REPLICON_FILES        = ["Timesheet Diary Notes Report 2026-06-25.csv"]
SERVICENOW_FILE       = "time_card_daily (10) (1).xlsx"
USER_MAPPING_APPROVED = "user_mapping_approved.xlsx"

TIMESTAMP  = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

AUTO_ACCEPT = 0.80
REVIEW_LOW  = 0.70

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 70)

print(f"Timestamp  : {TIMESTAMP}")
print(f"Output dir : {OUTPUT_DIR.resolve()}")
print(f"jellyfish  : {rec.HAS_JELLYFISH}")
print(f"rapidfuzz  : {rec.HAS_RAPIDFUZZ}")


## Load Data

In [ ]:
_rep_frames = []
for _path in REPLICON_FILES:
    _df = pd.read_csv(_path, dtype=str)
    _df["_source_file"] = Path(_path).name
    _rep_frames.append(_df)

replicon_raw = pd.concat(_rep_frames, ignore_index=True)
print(f"Replicon rows loaded    : {len(replicon_raw):,}")

_sn_frames = []
with pd.ExcelFile(SERVICENOW_FILE) as _xl:
    _sheet_names = _xl.sheet_names
    for _sheet in _sheet_names:
        _df = _xl.parse(_sheet)
        _df["_sheet"] = _sheet
        _sn_frames.append(_df)

sn_raw = pd.concat(_sn_frames, ignore_index=True)
print(f"ServiceNow rows loaded  : {len(sn_raw):,}  ({len(_sheet_names)} sheets: {_sheet_names})")


## Inspect

In [ ]:
print("REPLICON columns :", replicon_raw.columns.tolist())
print("REPLICON shape   :", replicon_raw.shape)
display(replicon_raw.head(4))

print("\nSERVICENOW columns :", sn_raw.columns.tolist())
print("SERVICENOW shape   :", sn_raw.shape)
display(sn_raw[["Date", "User", "User ID", "Project ID", "Time worked", "_sheet"]].head(4))

## Reconcile

In [ ]:
approved_mapping = pd.read_excel(USER_MAPPING_APPROVED) if Path(USER_MAPPING_APPROVED).exists() else None

results = rec.run(replicon_raw, sn_raw, approved_mapping, AUTO_ACCEPT, REVIEW_LOW)

user_mapping     = results["user_mapping"]
recon_table      = results["recon_table"]
recon_by_user    = results["recon_by_user"]
recon_by_month   = results["recon_by_month"]
exception_report = results["exception_report"]
summary_df       = results["summary"]

print(f"Window    : {results['replicon_min_date'].date()} to {results['replicon_max_date'].date()}")
print(f"Compared  : {len(recon_table):,} rows")
print(f"Variance  : {recon_table['variance'].sum():.2f} h")
print(f"Matched   : {user_mapping['match_status'].eq('auto_accepted').sum()} / {len(user_mapping)} users")
display(recon_by_user)


## Outputs

In [ ]:
user_mapping.to_excel(OUTPUT_DIR / f"user_mapping_{TIMESTAMP}.xlsx", index=False)
exception_report.to_excel(OUTPUT_DIR / f"exception_report_{TIMESTAMP}.xlsx", index=False)
summary_df.to_excel(OUTPUT_DIR / f"summary_{TIMESTAMP}.xlsx", index=False)

with pd.ExcelWriter(OUTPUT_DIR / f"reconciliation_{TIMESTAMP}.xlsx", engine="openpyxl") as _w:
    recon_table.to_excel(_w,   sheet_name="detail",  index=False)
    recon_by_user.to_excel(_w, sheet_name="by_user", index=False)
    for _month, _df in recon_by_month.items():
        _df.to_excel(_w, sheet_name=_month, index=False)

for p in sorted(OUTPUT_DIR.glob(f"*{TIMESTAMP}*")):
    print(f"Saved: {p.name}")


## Validate

In [ ]:
checks = []

def _check(desc, passed, detail=""):
    checks.append({"check": desc, "status": "PASS" if passed else "FAIL", "detail": detail})
    print(f"  [{'OK  ' if passed else 'FAIL'}] {desc}" + (f"  — {detail}" if detail else ""))


replicon     = results["replicon"]
sn_in_window = results["sn_in_window"]
replicon_agg = results["replicon_agg"]
sn_agg       = results["sn_agg"]

_raw_rows = len(replicon_raw.dropna(subset=["Entry Date", "User Name"]))
_check("No record loss after Replicon cleaning",
       len(replicon) == _raw_rows, f"{len(replicon)} after vs {_raw_rows} raw rows")

_rep = (replicon["hours"].sum(), replicon_agg["hours_replicon"].sum())
_check("Replicon hours tie before/after aggregation",
       abs(_rep[0] - _rep[1]) < 0.01, f"{_rep[0]:.2f} → {_rep[1]:.2f}")

_sn = (sn_in_window["hours"].sum(), sn_agg["hours_servicenow"].sum())
_check("ServiceNow hours tie before/after aggregation",
       abs(_sn[0] - _sn[1]) < 0.01, f"{_sn[0]:.2f} → {_sn[1]:.2f}")

_check("Blank Replicon hours → 0", replicon["hours"].isna().sum() == 0)
_check("Replicon dates parsed",    replicon["date"].isna().sum() == 0,
       f"{replicon['date'].isna().sum()} failures")
_check("ServiceNow dates parsed",  results["sn"]["date"].isna().sum() == 0)

_matched = user_mapping["match_status"].eq("auto_accepted").sum()
_check("User mapping coverage", True, f"{_matched}/{len(user_mapping)} auto-accepted")

display(pd.DataFrame(checks))


## Summary

In [ ]:
for _, row in summary_df.iterrows():
    print(f"  {row['metric']:<50} {row['value']}")
print(f"\nOutputs: {OUTPUT_DIR.resolve()}\n")

_needs_review = user_mapping[user_mapping["review_required"] == True]
if not _needs_review.empty:
    print(f"ACTION REQUIRED: {len(_needs_review)} user(s) need review\n")
    for _, r in _needs_review.iterrows():
        sn_id = r["servicenow_user_id"] if pd.notna(r["servicenow_user_id"]) else "NO MATCH"
        print(f"  {str(r['replicon_username']):<35} -> {str(sn_id):<30} (score={r['final_score']:.3f}, {r['match_method']})")
    print(f"\n  Fix: review output/user_mapping_*.xlsx, save as '{USER_MAPPING_APPROVED}', re-run.")
else:
    print("All users matched.")
